# API de Riesgo Crediticio (FastAPI)

In [1]:
import json
from pathlib import Path

import joblib
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

## Carga del modelo entrenado

In [2]:
MODEL_DIR = Path("model")
RISK_LABELS = {0: "bajo", 1: "medio", 2: "alto"}

model = joblib.load(MODEL_DIR / "knn_model.pkl")
scaler = joblib.load(MODEL_DIR / "scaler.pkl")
with open(MODEL_DIR / "features.json", "r", encoding="utf-8") as f:
    FEATURES = json.load(f)["features"]

print("Features esperadas por el modelo:", FEATURES)

Features esperadas por el modelo: ['Interest_Rate', 'Num_Credit_Inquiries', 'Outstanding_Debt', 'Delay_from_due_date', 'Num_Credit_Card']


## Validacion de entrada (Pydantic) y respuesta

In [3]:
class CreditApplication(BaseModel):
    interest_rate: float = Field(..., ge=0, description="Tasa de interes asignada")
    num_credit_inquiries: float = Field(..., ge=0, description="Numero de consultas de credito")
    outstanding_debt: float = Field(..., ge=0, description="Deuda pendiente")
    delay_from_due_date: float = Field(..., description="Dias de atraso promedio en pagos")
    num_credit_card: float = Field(..., ge=0, description="Numero de tarjetas de credito")


class PredictionResponse(BaseModel):
    risk_category: int
    risk_level: str

## Definicion de la API

In [4]:
app = FastAPI(title="API de Riesgo Crediticio")


@app.post("/predict", response_model=PredictionResponse)
def predict(application: CreditApplication):
    try:
        entrada = [[
            application.interest_rate,
            application.num_credit_inquiries,
            application.outstanding_debt,
            application.delay_from_due_date,
            application.num_credit_card,
        ]]
        entrada_escalada = scaler.transform(entrada)
        category = int(model.predict(entrada_escalada)[0])
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"Error al generar la prediccion: {exc}")

    return PredictionResponse(risk_category=category, risk_level=RISK_LABELS[category])


@app.get("/health")
def health():
    return {"status": "ok", "features": FEATURES}

## Ejecutar el servidor

In [ ]:
config = uvicorn.Config(app, host="127.0.0.1", port=8000)
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [11528]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:28319 - "POST /predict HTTP/1.1" 200 OK


C:\Users\lemor\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


INFO:     127.0.0.1:56586 - "POST /predict HTTP/1.1" 200 OK


C:\Users\lemor\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


INFO:     127.0.0.1:19015 - "POST /predict HTTP/1.1" 200 OK


C:\Users\lemor\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


INFO:     127.0.0.1:19026 - "POST /predict HTTP/1.1" 200 OK


C:\Users\lemor\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
